# Contrôle Qualité (QC) et Filtrage des Cellules (Nestorowa et al.)

Ce notebook réalise le contrôle qualité (QC) et le filtrage de cellules uniques (*scRNA-seq*) en reproduisant les seuils et la méthodologie de l'article **Nestorowa et al. (2016)** (GSE81682).
Il intègre également un module d'analyse fine des raisons d'exclusion par cellule et une comparaison croisée avec le jeu de données de **Chevalier et al.**.

## 1. Chargement des bibliothèques et initialisation

In [1]:
import numpy as np
import pandas as pd
import mygene

# Fixer la graine aléatoire pour la reproductibilité
np.random.seed(100)

## 2. Importation des données brutes et vérifications

On charge la matrice de comptage HTSeq et on effectue des vérifications de cohérence (doublons, valeurs manquantes, valeurs négatives).

In [ ]:
counts_path = "nestorowa_singlecell_raw_data_taken_from_original_paper_GSE.txt"

# Lecture du fichier TSV
raw_df = pd.read_csv(counts_path, sep="\t", index_col=0)
raw_counts = raw_df.astype(float)

# Vérifications de sécurité
assert not raw_counts.index.duplicated().any(), "Identifiants de gènes en double !"
assert not raw_counts.isna().any().any(), "Valeurs NA détectées !"
assert (raw_counts.values >= 0).all(), "Counts négatifs détectés !"

print(f"Lignes brutes (gènes/caractéristiques) : {raw_counts.shape[0]}")
print(f"Colonnes brutes (cellules)             : {raw_counts.shape[1]}")

## 3. Classification des types de lignes

Identification des gènes endogènes (`ENSMUSG`), des *spike-ins* ERCC et des lignes de résumé HTSeq (`__`).

In [ ]:
ids = raw_counts.index.to_series()

is_endogenous = ids.str.contains(r"^ENSMUSG", regex=True)
is_ercc = ids.str.contains(r"^ERCC[-_]", case=False, regex=True)
is_htseq = ids.str.contains(r"^__", regex=True)
is_unknown = ~(is_endogenous | is_ercc | is_htseq)

summary_df = pd.DataFrame({
    'Classe': ["ENSMUSG", "ERCC", "Résumé HTSeq", "Inconnu"],
    'Nombre': [is_endogenous.sum(), is_ercc.sum(), is_htseq.sum(), is_unknown.sum()]
})
print(summary_df)

if is_unknown.any():
    raise ValueError(f"Lignes inattendues : {list(ids[is_unknown])}")

## 4. Définition des gènes mitochondriaux (GRCm38)

Construction explicite des 37 identifiants Ensembl mitochondriaux de référence pour la souris.

In [ ]:
grcm38_mito_ensembl = [f"ENSMUSG000000{i}" for i in range(64336, 64362)] + \
                      [f"ENSMUSG000000{i}" for i in range(64363, 64373)] + \
                      ["ENSMUSG00000065947"]

assert len(grcm38_mito_ensembl) == 37

is_mito_full = is_endogenous & ids.isin(grcm38_mito_ensembl)
is_nuclear_full = is_endogenous & ~is_mito_full

print(f"Gènes mitochondriaux attendus : 37")
print(f"Gènes mitochondriaux présents dans la matrice : {is_mito_full.sum()}")

## 5. Séparation des matrices et seuils de QC

Séparation des sous-matrices (endogènes, ERCC, HTSeq) et définition des 4 critères de filtrage de Nestorowa et al.

In [ ]:
endogenous_counts = raw_counts.loc[is_endogenous]
ercc_counts = raw_counts.loc[is_ercc]
htseq_counts = raw_counts.loc[is_htseq]

# Seuils de contrôle qualité (Nestorowa et al. 2016)
MIN_NUCLEAR_READS = 200000
MIN_DETECTED_GENES = 4000
MAX_MITO_PCT = 10
MAX_ERCC_PCT = 50
NESTOROWA_REPORTED_CELLS = 1656

## 6. Reconstitution des métriques QC par cellule

Calcul des reads totaux, alignés, nucléaires, mitochondriaux, ERCC, ainsi que du nombre de gènes codants détectés via `MyGeneInfo`.

In [ ]:
def get_htseq_row(row_name):
    if row_name in htseq_counts.index:
        return htseq_counts.loc[row_name].values
    return np.zeros(raw_counts.shape[1])

raw_total_reads = raw_counts.sum(axis=0)
not_aligned = get_htseq_row("__not_aligned")
mapped_reads = raw_total_reads - not_aligned

nuclear_reads = raw_counts.loc[is_nuclear_full].sum(axis=0)
mitochondrial_reads = raw_counts.loc[is_mito_full].sum(axis=0)
ercc_reads = ercc_counts.sum(axis=0)
assigned_feature_reads = endogenous_counts.sum(axis=0) + ercc_reads

# Annotation des gènes codants pour des protéines
mg = mygene.MyGeneInfo()
endogenous_ids = ids[is_endogenous].tolist()
res = mg.getgenes(endogenous_ids, fields='type_of_gene', species='mouse')

gene_type_dict = {entry.get('query'): entry.get('type_of_gene') for entry in res}
is_protein_coding_list = [gene_type_dict.get(g) == 'protein-coding' for g in endogenous_ids]
is_protein_coding_endogenous = pd.Series(is_protein_coding_list, index=endogenous_counts.index)

# Nombre de gènes codants exprimés (> 0)
detected_genes = (endogenous_counts.loc[is_protein_coding_endogenous] > 0).sum(axis=0)

mito_pct_mapped = 100 * mitochondrial_reads / mapped_reads
ercc_pct_mapped = 100 * ercc_reads / mapped_reads

qc = pd.DataFrame({
    'cell': raw_counts.columns,
    'raw_total_reads': raw_total_reads,
    'mapped_reads': mapped_reads,
    'assigned_feature_reads': assigned_feature_reads,
    'nuclear_reads': nuclear_reads,
    'detected_genes': detected_genes,
    'mitochondrial_reads': mitochondrial_reads,
    'ercc_reads': ercc_reads,
    'mito_pct_mapped': mito_pct_mapped,
    'ercc_pct_mapped': ercc_pct_mapped
})

# Résumé statistique des métriques principales
qc[["mapped_reads", "nuclear_reads", "detected_genes", "mito_pct_mapped", "ercc_pct_mapped"]].describe()

## 7. Application des filtres QC et motifs d'échec

In [ ]:
qc['pass_nuclear'] = qc['nuclear_reads'] >= MIN_NUCLEAR_READS
qc['pass_genes'] = qc['detected_genes'] >= MIN_DETECTED_GENES
qc['pass_mito'] = qc['mito_pct_mapped'] < MAX_MITO_PCT
qc['pass_ercc'] = qc['ercc_pct_mapped'] < MAX_ERCC_PCT

qc['pass_all'] = (
    qc['pass_nuclear'] &
    qc['pass_genes'] &
    qc['pass_mito'] &
    qc['pass_ercc']
)

print(f"Cellules initiales        : {len(qc)}")
print(f"Cellules conservées (QC)  : {qc['pass_all'].sum()}")
print(f"Rapporté dans l'article   : {NESTOROWA_REPORTED_CELLS}")
print(f"Différence                : {qc['pass_all'].sum() - NESTOROWA_REPORTED_CELLS}\n")

## 8. Module Diagnostic : Raisons d'exclusion et comparaison avec Chevalier et al.

Ce module permet d'identifier l'étape d'exclusion exacte pour chaque cellule et de comparer les cellules conservées avec celles de Chevalier et al.

In [ ]:
def get_exclusion_reasons(qc_df):
    """
    Détermine pour chaque cellule si elle est conservée ou rejetée,
    et détaille précisément quels critères l'ont exclue.
    """
    reasons = []
    for _, row in qc_df.iterrows():
        if row['pass_all']:
            reasons.append("Retenue (PASS)")
        else:
            failed_criteria = []
            if not row['pass_nuclear']:
                failed_criteria.append("Reads nucléaires (< 200k)")
            if not row['pass_genes']:
                failed_criteria.append("Gènes codants (< 4k)")
            if not row['pass_mito']:
                failed_criteria.append("Mitochondrial (>= 10%)")
            if not row['pass_ercc']:
                failed_criteria.append("ERCC (>= 50%)")
            reasons.append(" | ".join(failed_criteria))
            
    qc_df['statut_exclusion'] = reasons
    return qc_df

# Application sur le DataFrame QC
qc = get_exclusion_reasons(qc)

print("--- Répartition globale des exclusions par critère --- ")
print(qc['statut_exclusion'].value_counts())

In [ ]:
def compare_and_diagnose_with_chevalier(qc_df, chevalier_filepath):
    """
    Compare les cellules entre ce QC et le jeu de données Chevalier et al.,
    puis explique pourquoi les cellules divergentes ont été exclues.
    """
    try:
        chevalier_df = pd.read_csv(chevalier_filepath, sep="\t", index_col=0)
        chevalier_cells = set(chevalier_df.columns)
        
        qc_pass_cells = set(qc_df.loc[qc_df['pass_all'], 'cell'])
        all_raw_cells = set(qc_df['cell'])
        
        in_qc_not_chevalier = qc_pass_cells - chevalier_cells
        in_chevalier_not_qc = chevalier_cells - qc_pass_cells
        
        print("\n==================================================")
        print("1. RAPPROCHEMENT DES CELLULES")
        print("==================================================")
        print(f"Cellules validées par ce QC         : {len(qc_pass_cells)}")
        print(f"Cellules dans Chevalier et al.      : {len(chevalier_cells)}")
        print(f"Cellules communes aux deux          : {len(qc_pass_cells.intersection(chevalier_cells))}")
        print(f"Cellules dans QC mais PAS Chevalier : {len(in_qc_not_chevalier)}")
        print(f"Cellules dans Chevalier mais PAS QC : {len(in_chevalier_not_qc)}")
        
        print("\n==================================================")
        print("2. DIAGNOSTIC DES CELLULES CHEVALIER EXCLUES PAR CE QC")
        print("==================================================")
        
        chevalier_exclues_df = qc_df[qc_df['cell'].isin(in_chevalier_not_qc)].copy()
        
        if not chevalier_exclues_df.empty:
            not_in_raw = in_chevalier_not_qc - all_raw_cells
            if not_in_raw:
                print(f"⚠️ {len(not_in_raw)} cellules de Chevalier n'existent pas dans la matrice brute initiale.")
            
            print("\nRaison(s) d'exclusion dans notre QC :")
            print(chevalier_exclues_df['statut_exclusion'].value_counts())
            
            print("\nDétail des métriques pour les cellules exclues de Chevalier :")
            cols_to_show = ['cell', 'nuclear_reads', 'detected_genes', 'mito_pct_mapped', 'ercc_pct_mapped', 'statut_exclusion']
            print(chevalier_exclues_df[cols_to_show].to_string(index=False))
        else:
            print("Toutes les cellules de Chevalier et al. ont été retenues par notre QC !")
            
        print("\n==================================================")
        print("3. CELLULES RETENUES PAR CE QC MAIS ABSENTES CHEZ CHEVALIER")
        print("==================================================")
        if in_qc_not_chevalier:
            qc_extra_df = qc_df[qc_df['cell'].isin(in_qc_not_chevalier)].copy()
            cols_to_show = ['cell', 'nuclear_reads', 'detected_genes', 'mito_pct_mapped', 'ercc_pct_mapped']
            print(qc_extra_df[cols_to_show].to_string(index=False))
        else:
            print("Aucune cellule supplémentaire retenue.")
            
        return {
            'in_qc_not_chevalier': list(in_qc_not_chevalier),
            'in_chevalier_not_qc': list(in_chevalier_not_qc),
            'chevalier_exclues_details': chevalier_exclues_df
        }
    except FileNotFoundError:
        print(f"\n❌ Le fichier {chevalier_filepath} est introuvable.")
        return None

# Exécution de l'analyse comparative
chevalier_path = "nestorowa_singlecell_raw_data_taken_from_bonesis_paper.tsv"
resultats_comparaison = compare_and_diagnose_with_chevalier(qc, chevalier_path)

## 9. Extraction de la matrice d'expression finale

In [ ]:
keep_cells = qc.loc[qc['pass_all'], 'cell']

endogenous_qc = endogenous_counts[keep_cells]
ercc_qc = ercc_counts[keep_cells]

# Retrait des caractéristiques non exprimées dans les cellules retenues
endogenous_qc = endogenous_qc.loc[endogenous_qc.sum(axis=1) > 0]
ercc_qc = ercc_qc.loc[ercc_qc.sum(axis=1) > 0]

print(f"Cellules retenues        : {endogenous_qc.shape[1]}")
print(f"Gènes endogènes filtrés : {endogenous_qc.shape[0]}")
print(f"ERCCs filtrés            : {ercc_qc.shape[0]}")